# ML Coding — AI Lab Instructor Notebook

*Deep Learning · Medium*

Practice feature creation from multi-table event logs (the most common real ML data task). You’ll build leakage-safe aggregate features using correct joins and time windows.

**Outcome.** Students can:
- design correct join keys and windows,
- avoid label leakage via time-aware filtering,
- compute aggregation features efficiently in pandas,
- reason about complexity and correctness.

6 tasks, 1 left as exercises (no source solution).

Runs top to bottom. Each task shows the assignment, then the instructor solution.

In [ ]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

def check(name: str, cond: bool):
    if not cond:
        raise AssertionError(f'Failed: {name}')
    print(f'OK: {name}')

rng = np.random.default_rng(0)

## Task 1: Dataset & Setup

# ML Coding — Multi-Table Log Joins (Student Lab)

You will generate multi-table logs and build leakage-safe aggregate features.

## Setup — Generate synthetic tables

Tables:
- users(user_id, country, signup_ts)
- sessions(session_id, user_id, session_ts)
- events(event_id, session_id, event_ts, page, event_type)
- labels(user_id, as_of_date, churn_30d)

In [ ]:
# No source solution — left as an exercise.

## Task 2: State join cardinalities
*Section: Join cardinalities*

## Section 1 — Join cardinalities

### Task 1.1: State join cardinalities

Write answers:
- users -> sessions is (1:N)
- sessions -> events is (1:N)
- users -> events is (1:N) via sessions

**Checkpoint:** What goes wrong with an accidental N:N join?

In [ ]:
def cardinality_answers():
    return {
        'users_to_sessions': '1:N (one user can have many sessions)',
        'sessions_to_events': '1:N (one session can have many events)',
        'users_to_events': '1:N via sessions (users->sessions->events)',
        'nn_failure_mode': 'N:N joins duplicate rows, explode counts, and inflate aggregates/metrics.',
    }

answers = cardinality_answers()
print(answers)

# Rationale: always reason about join cardinality before merging logs.


**Why this works.** always reason about join cardinality before merging logs.

## Section 1 — Join cardinalities

### Task 1.1: State join cardinalities

## Task 3: Build user-level features as-of `as_of_date`
*Section: Leakage-safe windowed aggregates*

## Section 2 — Leakage-safe windowed aggregates

### Task 2.1: Build user-level features as-of `as_of_date`

Features (examples):
- sessions_7d, sessions_30d
- events_7d, events_30d
- distinct_pages_30d

- Join sessions to labels on user_id
- Filter sessions to <= as_of_date (no future)
- Compute windowed counts

HINT: Use `as_of_date - Timedelta(days=7)` etc.

**Hints**
- Use `as_of_date - Timedelta(days=7)` etc.

In [ ]:
# Windowed session aggregates
user_anchor = labels[['user_id','as_of_date','churn_30d']].copy()
sess = sessions.merge(user_anchor[['user_id','as_of_date']], on='user_id', how='inner')
sess = sess[sess['session_ts'] <= sess['as_of_date']]
sess['in7'] = (sess['session_ts'] >= (sess['as_of_date'] - pd.to_timedelta(7, unit='D'))).astype(int)
sess['in30'] = (sess['session_ts'] >= (sess['as_of_date'] - pd.to_timedelta(30, unit='D'))).astype(int)
sessions_agg = sess.groupby('user_id').agg(sessions_7d=('in7','sum'), sessions_30d=('in30','sum')).reset_index()


**Why this works.** ## Section 2 — Leakage-safe windowed aggregates

### Task 2.1: Build user-level features as-of `as_of_date`

## Task 4: Event-based aggregates
*Section: Leakage-safe windowed aggregates*

### Task 2.2: Event-based aggregates

- Join events -> sessions -> user_id
- Filter events <= as_of_date
- Compute events_7d, events_30d, distinct_pages_30d per user

In [ ]:
# Windowed event aggregates
ev = events.merge(sessions[['session_id','user_id','session_ts']], on='session_id', how='left')
ev = ev.merge(user_anchor[['user_id','as_of_date']], on='user_id', how='inner')
ev = ev[ev['event_ts'] <= ev['as_of_date']]
ev['in7'] = (ev['event_ts'] >= (ev['as_of_date'] - pd.to_timedelta(7, unit='D'))).astype(int)
ev['in30'] = (ev['event_ts'] >= (ev['as_of_date'] - pd.to_timedelta(30, unit='D'))).astype(int)
ev30 = ev[ev['in30'].astype(bool)]
events_agg = ev.groupby('user_id').agg(events_7d=('in7','sum'), events_30d=('in30','sum')).reset_index()
distinct_pages_30d = ev30.groupby('user_id')['page'].nunique().rename('distinct_pages_30d').reset_index()
events_agg = events_agg.merge(distinct_pages_30d, on='user_id', how='left')
events_agg['distinct_pages_30d'] = events_agg['distinct_pages_30d'].fillna(0).astype(int)


## Task 5: Join features back to labels
*Section: Final training matrix*

## Section 3 — Final training matrix

### Task 3.1: Join features back to labels

- Left-join session + event aggregates onto labels
- Fill missing aggregates with 0
- Add a simple user feature (e.g., country one-hot)

**Hints**
- drop churn_30d and as_of_date

In [ ]:
df_train = user_anchor.merge(users[['user_id','country']], on='user_id', how='left')
df_train = df_train.merge(sessions_agg, on='user_id', how='left')
df_train = df_train.merge(events_agg, on='user_id', how='left')
for c in ['sessions_7d','sessions_30d','events_7d','events_30d','distinct_pages_30d']:
    df_train[c] = df_train[c].fillna(0).astype(int)
X = pd.get_dummies(df_train.drop(columns=['churn_30d', 'as_of_date']), columns=['country'], drop_first=True)
y = df_train['churn_30d'].values



**Why this works.** ## Section 3 — Final training matrix

### Task 3.1: Join features back to labels

## Task 6: Train a quick baseline model
*Section: Sanity model*

## Section 4 — Sanity model

### Task 4.1: Train a quick baseline model

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(X.values, y, test_size=0.3, random_state=0, stratify=y)
clf = LogisticRegression(max_iter=2000)
clf.fit(Xtr, ytr)
p = clf.predict_proba(Xte)[:, 1]
print('roc_auc', roc_auc_score(yte, p))

# Rationale: end-to-end sanity check; the main goal is correctness + leakage safety.



**Why this works.** end-to-end sanity check; the main goal is correctness + leakage safety.

## Section 4 — Sanity model

### Task 4.1: Train a quick baseline model